# Advertising Sales Forecasting
This notebook performs EDA, feature engineering, and regression model training on the Advertising dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

In [ ]:
data = pd.read_csv('../data/Advertising.csv')
data = data.drop(columns=['Unnamed: 0'], errors='ignore')
data['Total_Budget'] = data[['TV', 'Radio', 'Newspaper']].sum(axis=1)
data['TV_pct'] = data['TV'] / data['Total_Budget']
data['Radio_pct'] = data['Radio'] / data['Total_Budget']
data['Digital_pct'] = data['Newspaper'] / data['Total_Budget']
data.head()

In [ ]:
sns.pairplot(data[['TV', 'Radio', 'Newspaper', 'Sales']])
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
X = data[['TV', 'Radio', 'Newspaper']]
y = data['Sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
print('Linear Regression R2:', r2_score(y_test, y_pred_lr))
print('Linear Regression MAE:', mean_absolute_error(y_test, y_pred_lr))

In [ ]:
rf = RandomForestRegressor(random_state=42)
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 4, 6],
}
grid = GridSearchCV(rf, param_grid, cv=5, scoring='neg_mean_absolute_error')
grid.fit(X_train_scaled, y_train)
best_rf = grid.best_estimator_
y_pred_rf = best_rf.predict(X_test_scaled)
print('Random Forest R2:', r2_score(y_test, y_pred_rf))
print('Random Forest MAE:', mean_absolute_error(y_test, y_pred_rf))
print('Best RF params:', grid.best_params_)

In [ ]:
joblib.dump(best_rf, '../models/regression_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')